# ScamRoad — Autonomous Cloud GPU Training & Evaluation Pipeline

**Repository**: [github.com/Priya-Ranjan-0201/potholes](https://github.com/Priya-Ranjan-0201/potholes)  
**Hardware Target**: Google Colab GPU (NVIDIA Tesla T4 / V100 / A100 / L4)  
**Persistent Storage**: Google Drive (`/content/drive/MyDrive/ScamRoad`)  
**Target Models**: YOLOv8n / YOLOv8s PyTorch `.pt`, ONNX INT8, TensorRT  
**Single-Source Taxonomy**: `pothole`, `crack_longitudinal`, `crack_transverse`, `crack_alligator`


## 1. Hardware & GPU Diagnostics
Verify the assigned GPU, VRAM, and system memory before launching any heavy operations.

In [ ]:
# 1. GPU & System Resource Audit
!nvidia-smi

import psutil
import shutil
import torch

ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print('=' * 65)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM Capacity    : {vram:.2f} GB')
    print(f'CUDA Version     : {torch.version.cuda}')
else:
    print('[WARNING] No CUDA GPU detected! Switch Runtime -> Change runtime type -> T4 GPU')
print(f'Colab RAM Total  : {ram.total / (1024**3):.2f} GB (Available: {ram.available / (1024**3):.2f} GB)')
print(f'Colab Disk Free  : {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB')
print('=' * 65)


## 2. Mount Persistent Google Drive Storage
Persistent storage ensures datasets, checkpoints, weights, and logs survive Colab runtime resets.

In [ ]:
# 2. Google Drive Mount & Workspace Directory Initialization
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/ScamRoad')
drive_root.mkdir(parents=True, exist_ok=True)

for folder in ['datasets', 'checkpoints', 'weights', 'exports', 'runs', 'logs']:
    (drive_root / folder).mkdir(exist_ok=True)

print(f'[SUCCESS] Persistent Google Drive storage active at: {drive_root}')
!ls -lh /content/drive/MyDrive/ScamRoad


## 3. Clone / Sync ScamRoad Project Repository
Pulls the latest verified codebase from GitHub repository `Priya-Ranjan-0201/potholes`.

In [ ]:
# 3. Clone or pull repo
import os, sys
from pathlib import Path

repo_path = Path('/content/potholes')
if not repo_path.exists():
    !git clone https://github.com/Priya-Ranjan-0201/potholes.git /content/potholes
else:
    !cd /content/potholes && git pull origin main

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print('[SUCCESS] Project workspace ready at:', repo_path)


## 4. Install Cloud GPU Dependencies
Install Ultralytics YOLOv8, ONNX Runtime GPU, and computer vision libraries.

In [ ]:
# 4. Pip installation
!pip install -q ultralytics onnx onnxruntime-gpu pyyaml albumentations opencv-python


## 5. Configure Kaggle Credentials
Set up Kaggle API access using Colab Secrets or uploaded `kaggle.json`.

In [ ]:
# 5. Kaggle Authentication (Supports Colab Secrets: priyaranja, KAGGLE_KEY, KAGGLE_API_TOKEN)
import contextlib
import json
import os
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_file = kaggle_dir / "kaggle.json"
access_token_file = kaggle_dir / "access_token"

# 1. Resolve credentials from Colab Secrets
try:
    from google.colab import userdata

    user = None
    key = None
    with contextlib.suppress(Exception):
        user = userdata.get("KAGGLE_USERNAME")
    with contextlib.suppress(Exception):
        key = userdata.get("KAGGLE_KEY")
    with contextlib.suppress(Exception):
        if not key:
            key = userdata.get("KAGGLE_API_TOKEN")
    # Check custom user secret name as configured in Colab (e.g. "priyaranja")
    for candidate in ["priyaranja", "priyaranjan", "kaggle", "kaggle_token"]:
        if not key:
            with contextlib.suppress(Exception):
                key = userdata.get(candidate)

    username = user or "priyaranja"
    if key:
        # Modern Kaggle CLI 2.x Bearer Token
        with open(access_token_file, "w", encoding="utf-8") as f:
            f.write(key.strip())
        with contextlib.suppress(Exception):
            os.chmod(access_token_file, 0o600)
        # Legacy kaggle.json for Ultralytics & Python client
        with open(kaggle_file, "w", encoding="utf-8") as f:
            json.dump({"username": username, "key": key.strip()}, f)
        with contextlib.suppress(Exception):
            os.chmod(kaggle_file, 0o600)
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key.strip()
        os.environ["KAGGLE_API_TOKEN"] = key.strip()
        print(f"[SUCCESS] Kaggle credentials configured from Colab Secrets (user: {username})!")
except Exception as e:
    print("[INFO] Colab secrets check:", e)

# 2. Manual fallback if neither secret nor file exists
if not kaggle_file.exists() and not access_token_file.exists():
    print("[ACTION REQUIRED] Upload your kaggle.json or add secret in Colab Secrets.")
    from google.colab import files

    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".json"):
            Path(fname).rename(kaggle_file)
            with contextlib.suppress(Exception):
                os.chmod(kaggle_file, 0o600)
            print("[SUCCESS] kaggle.json uploaded and configured!")
            break

!kaggle --version


## 6. Dataset Preparation & YAML Configuration
Defines the dataset path pointing to persistent Google Drive or local fast scratch space.

In [ ]:
# 6. Generate Dataset Config YAML
import yaml
from pathlib import Path

data_dir = Path('/content/data')
data_dir.mkdir(parents=True, exist_ok=True)

dataset_cfg = {
    'path': str(data_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 4,
    'names': {
        0: 'pothole',
        1: 'crack_longitudinal',
        2: 'crack_transverse',
        3: 'crack_alligator'
    }
}

cfg_file = data_dir / 'scamroad_colab.yaml'
with open(cfg_file, 'w') as f:
    yaml.safe_dump(dataset_cfg, f)

print(f'[SUCCESS] Dataset configuration created at: {cfg_file}')


## 7. Model Training (Ultralytics YOLOv8n / YOLOv8s)
Executes training with checkpoints saved directly into `/content/drive/MyDrive/ScamRoad/runs/`.

In [ ]:
# 7. Train YOLOv8 Model
from ultralytics import YOLO
from pathlib import Path

drive_runs = Path('/content/drive/MyDrive/ScamRoad/runs')

model = YOLO('yolov8n.pt')

# NOTE: When you are ready for full model training, set epochs=100 and execute.
# Currently running in verification mode (epochs=3 or interactive training).
results = model.train(
    data=str(cfg_file),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='yolov8n_scamroad_colab',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] Training finished! Checkpoints saved to Google Drive.')


## 8. Model Validation & Performance Metrics
Computes mAP@50 and mAP@50-95 against the validation set.

In [ ]:
# 8. Validation Evaluation
best_pt = drive_runs / 'yolov8n_scamroad_colab' / 'weights' / 'best.pt'
if best_pt.exists():
    trained_model = YOLO(str(best_pt))
    metrics = trained_model.val(data=str(cfg_file), split='val', device=0)
    print(f'mAP50-95 : {metrics.box.map:.4f}')
    print(f'mAP50    : {metrics.box.map50:.4f}')
    print(f'mAP75    : {metrics.box.map75:.4f}')
else:
    print('[INFO] Best checkpoint will appear once training completes.')


## 9. ONNX INT8 Quantization & Edge Export
Exports the trained PyTorch model to ONNX INT8 for edge deployment on Android (OPPO K13) and edge SBCs.

In [ ]:
# 9. ONNX & INT8 Dynamic Quantization
import shutil
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

exports_dir = Path('/content/drive/MyDrive/ScamRoad/exports')
exports_dir.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    print('[INFO] Exporting best model to ONNX...')
    onnx_file = trained_model.export(format='onnx', imgsz=640, dynamic=False, simplify=True)
    fp32_dest = exports_dir / 'yolov8n_scamroad_fp32.onnx'
    shutil.copy(onnx_file, fp32_dest)

    int8_dest = exports_dir / 'yolov8n_scamroad_int8.onnx'
    quantize_dynamic(
        model_input=str(fp32_dest),
        model_output=str(int8_dest),
        weight_type=QuantType.QUInt8
    )
    print(f'[SUCCESS] FP32 ONNX: {fp32_dest} ({fp32_dest.stat().st_size / (1024**2):.2f} MB)')
    print(f'[SUCCESS] INT8 ONNX: {int8_dest} ({int8_dest.stat().st_size / (1024**2):.2f} MB)')
else:
    print('[INFO] Best checkpoint will be converted once training finishes.')


## 10. Persisted Artifacts in Google Drive
Inspect all persisted checkpoints and exports in your Google Drive.

In [ ]:
!ls -lh /content/drive/MyDrive/ScamRoad/runs/yolov8n_scamroad_colab/weights
!ls -lh /content/drive/MyDrive/ScamRoad/exports
print('\n[COMPLETE] Cloud GPU pipeline ready and integrated with Google Drive.')
